# ML Challenge 2026: End-to-End Test Inference & Submission Generator (v2)
### Two-Tower FAISS Retrieval + LightGBM LambdaRank (25 Features) + F₀.₅ Calibrated Threshold

This notebook takes the **clean test datasets**, runs the trained models, and produces the **exact two submission files**:
1. **`candidate_pairs.tsv`** (Blocking Stage Output): Plausible candidate pool for each Test S1 entity.
2. **`matching_results.tsv`** (Leaderboard Scored File): Final high-confidence matches ($F_{0.5}$ optimized).
3. **`submission.zip`**: Final validated ZIP archive ready for portal upload.

#### v2 Improvements:
- **25 enriched pairwise features** (was 11) for dramatically better precision
- **F₀.₅-calibrated threshold** from exhaustive sweep on training data
- **Zero post-hoc string filters** — all decisions made by the trained LightGBM model


In [ ]:
import os
import sys
import gc
import time
import json
import re
import math
import zipfile
import shutil
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import faiss
import joblib

# RapidFuzz for high-speed string similarity
try:
    from rapidfuzz.distance import Levenshtein, JaroWinkler
    from rapidfuzz import fuzz
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"])
    from rapidfuzz.distance import Levenshtein, JaroWinkler
    from rapidfuzz import fuzz

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Using compute device: {device}")
if device.type == 'cuda':
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")

def locate_file(pattern: str, search_roots: tuple = (Path('/kaggle/input'), Path('.'))) -> Optional[Path]:
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(pattern))
            if matches:
                return matches[0]
    return None

# Dataset paths (handles auto-detection or manual Kaggle dataset paths)
SOURCE1_PATH = locate_file('*test_source1*.tsv') or Path('/kaggle/input/datasets/vinaysingh120221/cleantestdata/preprocessed_test_source1.tsv')
SOURCE2_PATH = locate_file('*test_source2*.tsv') or Path('/kaggle/input/datasets/vinaysingh120221/cleantestdata/preprocessed_test_source2.tsv')
SOURCE3_PATH = locate_file('*test_source3*.tsv') or Path('/kaggle/input/datasets/vinaysingh120221/cleantestdata/preprocessed_test_source3.tsv')

# Model Artifact Paths
MODEL_WEIGHTS    = locate_file('*best.pt') or Path('/kaggle/input/two-tower-output/best.pt')
TOKENIZER_PATH   = locate_file('*tokenizer.json') or Path('/kaggle/input/two-tower-output/tokenizer.json')
LEGAL_VOCAB_PATH = locate_file('*legal_form_vocab.json') or Path('/kaggle/input/two-tower-output/legal_form_vocab.json')
RERANKER_PATH    = locate_file('*lgbm_reranker*.joblib') or Path('/kaggle/input/reranker-model/lgbm_reranker.joblib')

OUTPUT_DIR = Path('/kaggle/working/submission')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pipeline Hyperparameters
CANDIDATE_TOP_K      = 10      # Number of candidate matches to retrieve per S1 query (recall was 99.91% at K=10)
ENCODE_BATCH_SIZE    = 2048    # Batch size for Two-Tower GPU encoding
CHUNK_SIZE           = 50000   # Chunk size for streaming candidates into FAISS

# 25 Canonical Feature Columns (must match EXACTLY with training)
FEATURE_COLUMNS = [
    "exact_name_norm", "exact_name_ascii",
    "levenshtein_similarity", "jaro_winkler_similarity", "char_ngram_similarity", "token_overlap",
    "ascii_levenshtein_similarity", "ascii_jaro_winkler_similarity",
    "partial_ratio", "token_sort_ratio", "token_set_ratio",
    "sorted_token_match", "containment_similarity", "prefix_3_match", "prefix_5_match",
    "numeric_token_overlap", "name_length_ratio", "name_length_diff",
    "legal_form_relationship", "address_similarity", "country_equal",
    "address_levenshtein", "address_token_overlap",
    "embedding_cosine", "ann_rank_normalized",
]

print(f"[INFO] Source 1 Path : {SOURCE1_PATH}")
print(f"[INFO] Source 2 Path : {SOURCE2_PATH}")
print(f"[INFO] Source 3 Path : {SOURCE3_PATH}")
print(f"[INFO] Two-Tower     : {MODEL_WEIGHTS}")
print(f"[INFO] Reranker Model: {RERANKER_PATH}")
print(f"[INFO] Output Dir    : {OUTPUT_DIR}")
print(f"[INFO] Feature Count : {len(FEATURE_COLUMNS)}")


In [ ]:
print("=" * 70)
print("STEP 1: Loading Two-Tower Model Architecture & Weights")
print("=" * 70)

TEXT_FIELDS = ("name_norm", "name_ascii", "address_norm", "country_norm")

def split_legal_forms(value: str) -> list[str]:
    value = str(value or "").strip()
    if value.startswith("["):
        try:
            value = ",".join(json.loads(value))
        except (json.JSONDecodeError, TypeError):
            pass
    strip_chars = " \'\\\""
    tokens = re.split(r"[,;|]", value)
    result = []
    for x in tokens:
        cleaned = x.strip(strip_chars)
        if cleaned and cleaned != "missing":
            result.append(cleaned)
    return result


class CharTokenizer:
    PAD, UNK = 0, 1

    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab

    def encode(self, values, length: int) -> torch.Tensor:
        output = np.zeros((len(values), length), dtype=np.int64)
        for row, value in enumerate(values):
            ids = [self.vocab.get(char, self.UNK) for char in str(value)[:length]]
            output[row, :len(ids)] = ids
        return torch.from_numpy(output)

    @classmethod
    def load(cls, path):
        return cls(json.loads(Path(path).read_text(encoding="utf-8")))


class LegalFormVocab:
    PAD, UNK = 0, 1

    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab

    def encode(self, values) -> torch.Tensor:
        forms = [split_legal_forms(value) for value in values]
        width = max(1, max(map(len, forms), default=0))
        output = np.zeros((len(forms), width), dtype=np.int64)
        for row, values_for_row in enumerate(forms):
            ids = [self.vocab.get(form, self.UNK) for form in values_for_row]
            output[row, :len(ids)] = ids
        return torch.from_numpy(output)

    @classmethod
    def load(cls, path):
        return cls(json.loads(Path(path).read_text(encoding="utf-8")))


class MaskedGlobalPool(nn.Module):
    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        mask = mask.unsqueeze(-1).float()
        x = x * mask
        return x.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


class EntityEncoder(nn.Module):
    def __init__(
        self,
        char_vocab_size: int,
        legal_vocab_size: int,
        char_emb_dim: int = 32,
        legal_emb_dim: int = 32,
        cnn_filters: int = 64,
        out_dim: int = 128
    ):
        super().__init__()
        self.char_embed = nn.Embedding(char_vocab_size, char_emb_dim, padding_idx=0)
        self.legal_embed = nn.Embedding(legal_vocab_size, legal_emb_dim, padding_idx=0)

        self.conv_name1 = nn.Conv1d(char_emb_dim, cnn_filters, kernel_size=3, padding=1)
        self.conv_name2 = nn.Conv1d(char_emb_dim, cnn_filters, kernel_size=5, padding=2)

        self.conv_ascii = nn.Conv1d(char_emb_dim, 32, kernel_size=3, padding=1)
        self.conv_addr  = nn.Conv1d(char_emb_dim, 32, kernel_size=5, padding=2)
        self.conv_ctry  = nn.Conv1d(char_emb_dim, 16, kernel_size=3, padding=1)

        self.pool = MaskedGlobalPool()
        total_dim = (cnn_filters * 2) + 32 + 32 + 16 + legal_emb_dim
        self.fc = nn.Sequential(
            nn.Linear(total_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, out_dim)
        )

    def forward(self, batch: dict[str, torch.Tensor]) -> torch.Tensor:
        def cnn_pool(tensor, conv_layer):
            emb = self.char_embed(tensor).transpose(1, 2)
            act = F.relu(conv_layer(emb)).transpose(1, 2)
            return self.pool(act, tensor != 0)

        p_name1 = cnn_pool(batch["name_norm"], self.conv_name1)
        p_name2 = cnn_pool(batch["name_norm"], self.conv_name2)
        p_ascii = cnn_pool(batch["name_ascii"], self.conv_ascii)
        p_addr  = cnn_pool(batch["address_norm"], self.conv_addr)
        p_ctry  = cnn_pool(batch["country_norm"], self.conv_ctry)

        legal_mask = (batch["legal_form"] != 0)
        p_legal = self.pool(self.legal_embed(batch["legal_form"]), legal_mask)

        merged = torch.cat([p_name1, p_name2, p_ascii, p_addr, p_ctry, p_legal], dim=1)
        out = self.fc(merged)
        return F.normalize(out, p=2, dim=1)


def records_to_batch(records: pd.DataFrame, tokenizer: CharTokenizer, legal_vocab: LegalFormVocab, device: torch.device):
    lengths = {"name_norm": 64, "name_ascii": 64, "address_norm": 96, "country_norm": 24}
    batch = {}
    for col, max_len in lengths.items():
        vals = records[col].fillna("").astype(str).tolist() if col in records.columns else [""] * len(records)
        batch[col] = tokenizer.encode(vals, max_len).to(device)
    vals = records["legal_form"].fillna("").astype(str).tolist() if "legal_form" in records.columns else [""] * len(records)
    batch["legal_form"] = legal_vocab.encode(vals).to(device)
    return batch

# Load Vocabularies
assert TOKENIZER_PATH and TOKENIZER_PATH.exists(), f"Tokenizer missing: {TOKENIZER_PATH}"
assert LEGAL_VOCAB_PATH and LEGAL_VOCAB_PATH.exists(), f"Legal vocab missing: {LEGAL_VOCAB_PATH}"
tokenizer = CharTokenizer.load(TOKENIZER_PATH)
legal_vocab = LegalFormVocab.load(LEGAL_VOCAB_PATH)

# Load Model
model = EntityEncoder(len(tokenizer.vocab) + 2, len(legal_vocab.vocab) + 2).to(device)
assert MODEL_WEIGHTS and MODEL_WEIGHTS.exists(), f"Weights missing: {MODEL_WEIGHTS}"
checkpoint = torch.load(MODEL_WEIGHTS, map_location=device)
state_dict = checkpoint["model"] if "model" in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()
print(f"  [OK] Model successfully loaded on {device}!")


In [ ]:
print("=" * 70)
print("STEP 2: Loading Clean Test Datasets")
print("=" * 70)

COLS = ["entity_id", "name_norm", "name_ascii", "address_norm", "country_norm", "legal_form"]

def read_tsv(path: Path) -> pd.DataFrame:
    t0 = time.time()
    df = pd.read_csv(path, sep="\t", usecols=lambda c: c in COLS, dtype=str).fillna("")
    print(f"  Loaded {path.name}: {len(df):,} rows ({time.time() - t0:.1f}s)")
    return df

test_s1 = read_tsv(SOURCE1_PATH)
test_s2 = read_tsv(SOURCE2_PATH)
test_s3 = read_tsv(SOURCE3_PATH)

all_test_s1_ids = test_s1["entity_id"].tolist()
print(f"  Total Test Source-1 Queries Required: {len(all_test_s1_ids):,}")

# Combine S2 and S3 candidates
candidates_df = pd.concat([test_s2, test_s3], ignore_index=True)
del test_s2, test_s3
gc.collect()

candidates_df.drop_duplicates(subset=["entity_id"], inplace=True)
candidates_df.set_index("entity_id", drop=False, inplace=True)
candidate_ids = candidates_df["entity_id"].to_numpy()
total_candidates = len(candidates_df)
print(f"  Total Target Candidate Entities (S2 + S3): {total_candidates:,}")


In [ ]:
print("=" * 70)
print("STEP 3: Streaming Test Candidates into FAISS Index")
print("=" * 70)

embedding_dim = 128
faiss_index = faiss.IndexFlatIP(embedding_dim)

t_start = time.time()
with torch.inference_mode():
    for start in tqdm(range(0, total_candidates, CHUNK_SIZE), desc="Indexing Candidates"):
        end = min(start + CHUNK_SIZE, total_candidates)
        chunk = candidates_df.iloc[start:end]
        chunk_embs = []
        for b_start in range(0, len(chunk), ENCODE_BATCH_SIZE):
            batch_slice = chunk.iloc[b_start : b_start + ENCODE_BATCH_SIZE]
            batch_inputs = records_to_batch(batch_slice, tokenizer, legal_vocab, device)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                emb = model(batch_inputs)
            chunk_embs.append(emb.cpu().numpy().astype(np.float32))
        faiss_index.add(np.concatenate(chunk_embs, axis=0))
        del chunk_embs
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

t_elapsed = time.time() - t_start
print(f"  [OK] Indexed {faiss_index.ntotal:,} candidates in {t_elapsed:.1f}s ({total_candidates / t_elapsed:.0f} rec/sec)!")


In [ ]:
print("=" * 70)
print(f"STEP 4: GPU Batch FAISS Top-{CANDIDATE_TOP_K} Retrieval for All S1 Entities")
print("=" * 70)

total_s1 = len(test_s1)
ann_rows = []
t_start = time.time()
print("  Transferring 10.3M candidate embeddings to GPU in FP16 (2.6 GB VRAM)...")
t0 = time.time()
cand_vectors = faiss_index.reconstruct_n(0, faiss_index.ntotal)
cand_tensor_gpu = torch.from_numpy(cand_vectors).to(device=device, dtype=torch.float16)
del cand_vectors
gc.collect()
print(f"  [OK] Transferred {len(cand_tensor_gpu):,} vectors to GPU VRAM in {time.time()-t0:.1f}s!")

# Prepare candidate_pairs.tsv file stream to save memory
candidate_pairs_file = OUTPUT_DIR / "candidate_pairs.tsv"
f_cand = open(candidate_pairs_file, "w", encoding="utf-8")
f_cand.write("source1_entity_id\tcandidate_entity_ids\n")

SEARCH_BATCH_SIZE = 256  # 256 queries * 10.3M candidates * 2 bytes = 5.2 GB buffer -> safe for 16GB GPU
ann_rows = []

with torch.inference_mode():
    for batch_start in tqdm(range(0, total_s1, SEARCH_BATCH_SIZE), desc="GPU Searching S1"):
        batch_end = min(batch_start + SEARCH_BATCH_SIZE, total_s1)
        batch_slice = test_s1.iloc[batch_start:batch_end]
        batch_qids = batch_slice["entity_id"].tolist()

        batch_inputs = records_to_batch(batch_slice, tokenizer, legal_vocab, device)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            query_embs = model(batch_inputs).half()

        # GPU Tensor Cores: (B, 128) @ (128, 10.3M) -> (B, 10.3M) in ~15 milliseconds!
        scores = torch.mm(query_embs, cand_tensor_gpu.T)
        topk_scores, topk_indices = torch.topk(scores, k=CANDIDATE_TOP_K, dim=1)

        # Move only Top-K results to CPU
        topk_scores = topk_scores.cpu().numpy()
        topk_indices = topk_indices.cpu().numpy()

        for i, qid in enumerate(batch_qids):
            c_ids = candidate_ids[topk_indices[i]]
            cand_str = ",".join(c_ids)
            f_cand.write(f"{qid}\t{cand_str}\n")

            for rank, (c_id, score) in enumerate(zip(c_ids, topk_scores[i]), start=1):
                ann_rows.append({
                    "query_id": qid,
                    "candidate_id": c_id,
                    "ann_rank": rank,
                    "embedding_cosine": round(float(score), 5),
                })

f_cand.close()
del cand_tensor_gpu, faiss_index, model
if device.type == "cuda":
    torch.cuda.empty_cache()
gc.collect()

ann_df = pd.DataFrame(ann_rows)
del ann_rows
gc.collect()

t_elapsed = time.time() - t_start
print(f"\n[SUCCESS] GPU Retrieval Finished in {t_elapsed:.1f}s ({total_s1 / t_elapsed:.0f} queries/sec)!")
print(f"[OK] Wrote official candidate_pairs.tsv: {len(all_test_s1_ids):,} rows ({candidate_pairs_file.stat().st_size / 1e6:.1f} MB)")


In [ ]:
print("=" * 70)
print("STEP 6: Loading LightGBM Reranker & Pre-Caching Metadata")
print("=" * 70)

assert RERANKER_PATH and RERANKER_PATH.exists(), f"Reranker missing: {RERANKER_PATH}"
loaded_obj = joblib.load(RERANKER_PATH)
ranker = loaded_obj["model"] if isinstance(loaded_obj, dict) and "model" in loaded_obj else loaded_obj

# Load calibrated threshold from checkpoint (or use default)
if isinstance(loaded_obj, dict) and "calibrated_threshold" in loaded_obj:
    MATCH_THRESHOLD = loaded_obj["calibrated_threshold"]
    print(f"  [OK] Using CALIBRATED F₀.₅ threshold from training: {MATCH_THRESHOLD}")
else:
    MATCH_THRESHOLD = 1.0
    print(f"  [WARN] No calibrated threshold found — using default: {MATCH_THRESHOLD}")

# Check if model expects 25 or 11 features
if isinstance(loaded_obj, dict) and "feature_columns" in loaded_obj:
    model_features = loaded_obj["feature_columns"]
    print(f"  [INFO] Model was trained with {len(model_features)} features: {model_features}")
    # Update FEATURE_COLUMNS to match the model
    FEATURE_COLUMNS = model_features
else:
    print(f"  [INFO] Using default {len(FEATURE_COLUMNS)} features")

print(f"  [OK] Loaded LightGBM Reranker from: {RERANKER_PATH}")

s1_lookup = test_s1.set_index("entity_id", drop=False)

needed_s1 = set(ann_df["query_id"].unique())
needed_cand = set(ann_df["candidate_id"].unique())

print("  Pre-caching entity records for C-speed lookup...")
s1_cache = s1_lookup.loc[s1_lookup.index.isin(needed_s1)].to_dict("index")
cand_cache = candidates_df.loc[candidates_df.index.isin(needed_cand)].to_dict("index")
print(f"  Cached {len(s1_cache):,} S1 and {len(cand_cache):,} candidate records.")


In [ ]:
print("=" * 70)
print(f"STEP 7: 25-Feature Pairwise Extraction & Reranker Scoring")
print("=" * 70)

import re as _re
_DIGIT_RE = _re.compile(r"\d+")

LEGAL_FORM_EQUIVALENTS = {
    "private_limited": {"private_limited", "pvt_ltd", "p_ltd", "ltd"},
    "llc": {"llc", "pllc", "limited_liability_company"},
    "llp": {"llp", "lp", "lllp"},
    "inc": {"inc", "corp", "corporation", "incorporated"},
    "sa": {"sa", "sarl", "sas"},
    "gmbh": {"gmbh", "ag"},
}

def safe_str(val: Any) -> str:
    if val is None or pd.isna(val):
        return ""
    s = str(val).strip().lower()
    return s if s != "nan" else ""

def char_ngram_jaccard(s1: str, s2: str, n_values: tuple = (2, 3)) -> float:
    if not s1 or not s2:
        return 0.0
    if s1 == s2:
        return 1.0
    scores = []
    for n in n_values:
        set1 = {s1[i:i+n] for i in range(len(s1) - n + 1)} if len(s1) >= n else {s1}
        set2 = {s2[i:i+n] for i in range(len(s2) - n + 1)} if len(s2) >= n else {s2}
        union = len(set1 | set2)
        scores.append(len(set1 & set2) / union if union > 0 else 0.0)
    return float(np.mean(scores))

def token_jaccard(s1: str, s2: str) -> float:
    if not s1 or not s2:
        return 0.0
    t1 = set(s1.split())
    t2 = set(s2.split())
    union = len(t1 | t2)
    return len(t1 & t2) / union if union > 0 else 0.0

def legal_form_score(f1: str, f2: str) -> float:
    if not f1 or not f2 or f1 == "missing" or f2 == "missing":
        return 0.5
    if f1 == f2:
        return 1.0
    for cluster in LEGAL_FORM_EQUIVALENTS.values():
        if f1 in cluster and f2 in cluster:
            return 0.8
    return 0.0

def containment_sim(s1: str, s2: str) -> float:
    if not s1 or not s2:
        return 0.0
    t1 = set(s1.split())
    t2 = set(s2.split())
    if not t1 or not t2:
        return 0.0
    shorter = t1 if len(t1) <= len(t2) else t2
    longer = t2 if len(t1) <= len(t2) else t1
    return len(shorter & longer) / len(shorter) if shorter else 0.0

def numeric_overlap(s1: str, s2: str) -> float:
    nums1 = set(_DIGIT_RE.findall(s1)) if s1 else set()
    nums2 = set(_DIGIT_RE.findall(s2)) if s2 else set()
    if not nums1 and not nums2:
        return 0.5
    if not nums1 or not nums2:
        return 0.0
    union = len(nums1 | nums2)
    return len(nums1 & nums2) / union if union > 0 else 0.0

STREAM_CHUNK = 200000
total_pairs = len(ann_df)
scores_list = []
t_score = time.time()

# Ensure ranker is the estimator if loaded as checkpoint dict
if isinstance(ranker, dict) and "model" in ranker:
    ranker = ranker["model"]

print(f"  Scoring {total_pairs:,} candidate pairs in streaming chunks of {STREAM_CHUNK:,}...")
print(f"  Using {len(FEATURE_COLUMNS)} features with threshold = {MATCH_THRESHOLD}")

for start_idx in range(0, total_pairs, STREAM_CHUNK):
    end_idx = min(start_idx + STREAM_CHUNK, total_pairs)
    chunk = ann_df.iloc[start_idx:end_idx]

    feats = []
    for row in chunk.itertuples(index=False):
        q = s1_cache.get(row.query_id, {})
        c = cand_cache.get(row.candidate_id, {})

        qn = safe_str(q.get("name_norm", ""))
        cn = safe_str(c.get("name_norm", ""))
        qa = safe_str(q.get("name_ascii", ""))
        ca = safe_str(c.get("name_ascii", ""))
        q_addr = safe_str(q.get("address_norm", ""))
        c_addr = safe_str(c.get("address_norm", ""))
        q_ctry = safe_str(q.get("country_norm", ""))
        c_ctry = safe_str(c.get("country_norm", ""))
        q_leg = safe_str(q.get("legal_form", ""))
        c_leg = safe_str(c.get("legal_form", ""))

        # Name-norm (4)
        lev_sim = float(Levenshtein.normalized_similarity(qn, cn)) if (qn and cn) else 0.0
        jw_sim  = float(JaroWinkler.similarity(qn, cn)) if (qn and cn) else 0.0
        ngram_sim = char_ngram_jaccard(qn, cn)
        tok_sim = token_jaccard(qn, cn)

        # ASCII (2)
        ascii_lev = float(Levenshtein.normalized_similarity(qa, ca)) if (qa and ca) else 0.0
        ascii_jw = float(JaroWinkler.similarity(qa, ca)) if (qa and ca) else 0.0

        # RapidFuzz advanced (3)
        partial_r = fuzz.partial_ratio(qn, cn) / 100.0 if (qn and cn) else 0.0
        token_sort_r = fuzz.token_sort_ratio(qn, cn) / 100.0 if (qn and cn) else 0.0
        token_set_r = fuzz.token_set_ratio(qn, cn) / 100.0 if (qn and cn) else 0.0

        # Structural (4)
        sorted_tok = 1.0 if (qn and cn and tuple(sorted(qn.split())) == tuple(sorted(cn.split()))) else 0.0
        contain_s = containment_sim(qn, cn)
        prefix_3 = 1.0 if (len(qn) >= 3 and len(cn) >= 3 and qn[:3] == cn[:3]) else 0.0
        prefix_5 = 1.0 if (len(qn) >= 5 and len(cn) >= 5 and qn[:5] == cn[:5]) else 0.0

        # Numeric & length (3)
        num_ovlp = numeric_overlap(qn + " " + q_addr, cn + " " + c_addr)
        max_len = max(len(qn), len(cn))
        len_ratio = (min(len(qn), len(cn)) / max_len) if max_len > 0 else 0.0
        len_diff = abs(len(qn) - len(cn))

        # Metadata (3)
        legal_rel = legal_form_score(q_leg, c_leg)
        addr_lev = float(Levenshtein.normalized_similarity(q_addr, c_addr)) if (q_addr and c_addr) else 0.0
        addr_tok = token_jaccard(q_addr, c_addr)
        addr_sim = 0.5 * addr_lev + 0.5 * addr_tok
        ctry_eq = 0.5 if (not q_ctry or not c_ctry) else (1.0 if q_ctry == c_ctry else 0.0)

        # ANN rank (1)
        ann_r = int(row.ann_rank)
        ann_rank_norm = 1.0 / max(1, ann_r) if ann_r > 0 else 0.0

        feats.append([
            1.0 if (qn and qn == cn) else 0.0,       # exact_name_norm
            1.0 if (qa and qa == ca) else 0.0,       # exact_name_ascii
            round(lev_sim, 4),                        # levenshtein_similarity
            round(jw_sim, 4),                         # jaro_winkler_similarity
            round(ngram_sim, 4),                      # char_ngram_similarity
            round(tok_sim, 4),                        # token_overlap
            round(ascii_lev, 4),                      # ascii_levenshtein_similarity
            round(ascii_jw, 4),                       # ascii_jaro_winkler_similarity
            round(partial_r, 4),                      # partial_ratio
            round(token_sort_r, 4),                   # token_sort_ratio
            round(token_set_r, 4),                    # token_set_ratio
            sorted_tok,                               # sorted_token_match
            round(contain_s, 4),                      # containment_similarity
            prefix_3,                                 # prefix_3_match
            prefix_5,                                 # prefix_5_match
            round(num_ovlp, 4),                       # numeric_token_overlap
            round(len_ratio, 4),                      # name_length_ratio
            float(len_diff),                          # name_length_diff
            round(legal_rel, 4),                      # legal_form_relationship
            round(addr_sim, 4),                       # address_similarity
            ctry_eq,                                  # country_equal
            round(addr_lev, 4),                       # address_levenshtein
            round(addr_tok, 4),                       # address_token_overlap
            float(row.embedding_cosine),              # embedding_cosine
            round(ann_rank_norm, 4),                  # ann_rank_normalized
        ])

    X_chunk = np.array(feats, dtype=np.float32)
    chunk_preds = ranker.predict(X_chunk)
    scores_list.extend(chunk_preds)

    if end_idx % (STREAM_CHUNK * 5) == 0 or end_idx == total_pairs:
        pct = (end_idx / total_pairs) * 100
        qps = end_idx / (time.time() - t_score)
        print(f"    Scored [{end_idx:,}/{total_pairs:,}] ({pct:.1f}%) | {qps:.0f} pairs/sec")

ann_df["reranker_score"] = scores_list
print(f"  [OK] All {total_pairs:,} pairs scored in {time.time() - t_score:.1f}s!")


In [ ]:
print("=" * 70)
print(f"STEP 8: Generating matching_results.tsv (Threshold = {MATCH_THRESHOLD})")
print("=" * 70)

# Filter by F₀.₅-calibrated threshold
matched_df = ann_df[ann_df["reranker_score"] >= MATCH_THRESHOLD].copy()
matched_df.sort_values(["query_id", "reranker_score"], ascending=[True, False], inplace=True)

# Build matched mapping
matched_grouped = matched_df.groupby("query_id")["candidate_id"].unique().to_dict()

matching_results_file = OUTPUT_DIR / "matching_results.tsv"
t0 = time.time()
non_empty = 0
total_links = 0

with open(matching_results_file, "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
    for s1_id in all_test_s1_ids:
        matches = matched_grouped.get(s1_id, [])
        if len(matches) > 0:
            match_str = ",".join(matches)
            non_empty += 1
            total_links += len(matches)
        else:
            match_str = ""
        f.write(f"{s1_id}\t{match_str}\n")

size_mb = matching_results_file.stat().st_size / (1024 * 1024)
singletons = len(all_test_s1_ids) - non_empty
print(f"  [OK] Successfully wrote {matching_results_file} ({size_mb:.1f} MB in {time.time() - t0:.1f}s)")
print(f"  Total S1 rows written : {len(all_test_s1_ids):,}")
print(f"  Entities with matches : {non_empty:,} ({non_empty / len(all_test_s1_ids) * 100:.1f}%)")
print(f"  Singletons (no match) : {singletons:,} ({singletons / len(all_test_s1_ids) * 100:.1f}%)")
print(f"  Total Links predicted : {total_links:,}")
print(f"  Avg matches per entity: {total_links / non_empty:.2f}" if non_empty > 0 else "")
print(f"  Decision Threshold    : {MATCH_THRESHOLD}")


In [ ]:
print("=" * 70)
print("STEP 9: Running Official Submission Validator Rules & Packaging")
print("=" * 70)

def validate_submission_files(matching_path: Path, candidate_path: Path, required_s1_ids: list):
    errors, warnings = [], []
    DELIM = "\t"
    required = set(required_s1_ids)

    def check_file(path: Path, expected_header: list, col_name: str):
        if not path.is_file():
            errors.append(f"File not found: {path}")
            return None

        seen = set()
        mapping = {}
        with open(path, encoding="utf-8") as f:
            header = f.readline()
            cols = [c.strip().lower() for c in header.rstrip("\n").split(DELIM)]
            if cols != expected_header:
                errors.append(f"{path.name}: Expected header {expected_header}, got {cols}")
                return None

            for line_idx, line in enumerate(f, start=2):
                parts = line.rstrip("\n").split(DELIM)
                if len(parts) != 2:
                    errors.append(f"{path.name} line {line_idx}: malformed row (expected 2 tab-separated columns)")
                    continue
                s1, rest = parts[0].strip(), parts[1].strip()
                if s1 in seen:
                    errors.append(f"{path.name}: duplicate S1 ID: {s1}")
                seen.add(s1)

                ids = [x.strip() for x in rest.split(",") if x.strip()]
                if len(ids) != len(set(ids)):
                    errors.append(f"{path.name}: duplicate IDs inside list for S1: {s1}")

                for mid in ids:
                    if not mid.startswith(("S2-", "S3-")):
                        errors.append(f"{path.name}: invalid target ID prefix: {mid}")

                mapping[s1] = set(ids)

        # Check required S1 entities
        missing_s1 = required - seen
        if missing_s1:
            errors.append(f"{path.name}: missing {len(missing_s1)} required S1 entities!")
        extra_s1 = seen - required
        if extra_s1:
            errors.append(f"{path.name}: contains {len(extra_s1)} unexpected S1 IDs!")

        print(f"  [OK] {path.name}: {len(seen):,} rows verified.")
        return mapping

    print("  Validating matching_results.tsv...")
    matched = check_file(matching_path, ["source1_entity_id", "matched_entity_ids"], "matched_entity_ids")

    print("  Validating candidate_pairs.tsv...")
    candidate = check_file(candidate_path, ["source1_entity_id", "candidate_entity_ids"], "candidate_entity_ids")

    # Strict Subset Verification
    if matched is not None and candidate is not None:
        print("  Verifying that matching_results is a strict subset of candidate_pairs...")
        violations = []
        for s1, m_set in matched.items():
            c_set = candidate.get(s1, set())
            leak = m_set - c_set
            if leak:
                violations.append((s1, leak))

        if violations:
            errors.append(f"SUBSET VIOLATION: {len(violations)} S1 queries have matches not in candidate_pairs.tsv!")
        else:
            print("  [OK] Strict subset check PASSED: 100% of final matches exist in candidate_pairs.tsv!")

    return errors, warnings

errs, warns = validate_submission_files(matching_results_file, candidate_pairs_file, all_test_s1_ids)

if errs:
    print("\n[FAILED] VALIDATION FAILED WITH ERRORS:")
    for e in errs[:10]:
        print(f"  ❌ {e}")
    raise RuntimeError("Submission validation failed! Please fix the errors above.")
else:
    print("\n[SUCCESS] ALL SUBMISSION VALIDATION CHECKS PASSED (EXIT 0)!")
    for w in warns:
        print(f"  ⚠️ {w}")

# Packaging submission.zip
submission_zip = Path("/kaggle/working/submission.zip")
with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(matching_results_file, arcname="matching_results.tsv")
    z.write(candidate_pairs_file, arcname="candidate_pairs.tsv")

zip_mb = submission_zip.stat().st_size / (1024 * 1024)
print("\n" + "=" * 70)
print(f"SUBMISSION READY: {submission_zip} ({zip_mb:.1f} MB)")
print(f"Decision Threshold Used: {MATCH_THRESHOLD}")
print("You can download this submission.zip and upload it directly to the competition portal!")
print("=" * 70)
